<img src="logos/logo_Facyt.png"
     width="250"
     style="display: block; margin-left: auto; margin-right: auto;">

# Pokemon - Data Preparation & Feature Engineering

This notebook builds a professional data preparation phase for Pokemon battle modeling.

Key principles:
- Define the battle-level target (`first_wins`).
- Split train/test before any pipeline fitting.
- Avoid leakage (do not use `Winner`, `WinRate`, `Wins`, `n_combats`).
- Use `Pipeline` and `ColumnTransformer` for reproducible transformations.
- Persist artifacts for the next modeling notebook.


## Imports and setup


In [ ]:
from pathlib import Path
from typing import Dict, List
import json
import random

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Aligned with EDA for cross-phase reproducibility.
RANDOM_STATE = 29
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

TEST_SIZE = 0.20
DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../artifacts")
TARGET_COL = "first_wins"

GLOBAL_CONFIG: Dict[str, object] = {
    "id_col": "#",
    "target_col": TARGET_COL,
    "leakage_columns": ["Winner", "WinRate", "Wins", "n_combats"],
    "stats_cols": ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"],
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

### Setup interpretation

- This block sets the random seed and global parameters to guarantee reproducibility across EDA, preparation, and modeling.
- The explicit definition of leakage-risk columns establishes a shared policy that drives all downstream exclusions.

## Data loading
Load base Pokedex and battle tables.


In [ ]:
pokemon_path = DATA_DIR / "pokemon.csv"
combats_path = DATA_DIR / "combats.csv"

pokemon_df = pd.read_csv(pokemon_path)
combats_df = pd.read_csv(combats_path)

print(f"pokemon_df shape: {pokemon_df.shape}")
print(f"combats_df shape: {combats_df.shape}")
display(pokemon_df.head(3))
display(combats_df.head(3))


### Data loading interpretation

- The initial dimensions confirm the operational scale of the problem and bound the compute cost of preparation.
- This verification connects with the EDA audit: before building features, base sources are validated for completeness and readability.

## Minimal audit and anti-leakage rules
Validate the minimum schema and declare forbidden training variables.


In [ ]:
required_pokemon_cols = {
    "#", "Name", "Type 1", "Type 2", "HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Legendary"
}
required_combats_cols = {"First_pokemon", "Second_pokemon", "Winner"}

missing_pokemon_cols = sorted(required_pokemon_cols.difference(pokemon_df.columns))
missing_combats_cols = sorted(required_combats_cols.difference(combats_df.columns))

if missing_pokemon_cols:
    raise ValueError(f"Missing columns in pokemon_df: {missing_pokemon_cols}")
if missing_combats_cols:
    raise ValueError(f"Missing columns in combats_df: {missing_combats_cols}")

LEAKAGE_COLUMNS = {"Winner", "WinRate", "Wins", "n_combats"}
print("Leakage columns forbidden as features:", sorted(LEAKAGE_COLUMNS))

pokemon_ids = set(pokemon_df["#"].astype(int))
combat_ids = set(combats_df["First_pokemon"]).union(set(combats_df["Second_pokemon"]))
coverage = len(combat_ids.intersection(pokemon_ids)) / len(combat_ids)
print(f"Coverage of battle IDs within Pokedex: {coverage * 100:.2f}%")


### Anti-leakage rules interpretation

- Near-100% ID coverage implies battle-level joins are reliable.
- Declaring `Winner`, `WinRate`, `Wins`, and `n_combats` as forbidden early keeps preparation aligned with the core EDA leakage conclusion.

## Structural cleaning
Remove exact duplicates and normalize base fields.


In [ ]:
initial_rows = len(combats_df)
combats_df = combats_df.drop_duplicates(subset=["First_pokemon", "Second_pokemon", "Winner"]).copy()
removed_duplicates = initial_rows - len(combats_df)

pokemon_df = pokemon_df.copy()
pokemon_df["Type 2"] = pokemon_df["Type 2"].fillna("None")
pokemon_df["is_mega"] = pokemon_df["Name"].str.contains(r"Mega|Primal", case=False, na=False).astype(int)
pokemon_df["Legendary"] = pokemon_df["Legendary"].astype(int)

print(f"Original battles: {initial_rows:,}")
print(f"Battles after deduplication: {len(combats_df):,}")
print(f"Duplicates removed: {removed_duplicates:,}")


### Structural cleaning interpretation

- Deduplication reduces artificial dependency across samples and improves out-of-sample evaluation validity.
- Normalizing `Type 2`, `Legendary`, and `is_mega` prepares a consistent schema for relational feature engineering.

## Feature engineering functions
Build battle-level relational features to represent both contenders and explicit relative advantage patterns.

Design choices:
- numeric relative signal: `diff_*` and `abs_diff_*`,
- categorical relational signal: unordered type-pair interactions,
- binary context signal: same-type, same-generation, legendary/mega combinations,
- strict integrity checks after joins.

In [ ]:
STATS_COLS: List[str] = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]


def _make_unordered_pair(a: str, b: str) -> str:
    """Return a stable unordered pair key for two categorical values."""
    return "|".join(sorted([str(a), str(b)]))


def build_pokedex_lookup(pokemon_frame: pd.DataFrame) -> pd.DataFrame:
    """Create an ID-indexed lookup table with normalized Pokemon attributes.

    The function computes `stats_total` and sets Pokemon ID as index to support
    deterministic and efficient mapping during battle-level feature generation.
    """
    lookup = pokemon_frame.copy()
    lookup["stats_total"] = lookup[STATS_COLS].sum(axis=1)
    return lookup.set_index("#")


def build_battle_level_dataset(combats_frame: pd.DataFrame, pokedex_lookup: pd.DataFrame) -> pd.DataFrame:
    """Build a battle-level modeling dataset with relational features.

    Steps:
    1. Create battle target (`first_wins`) and dependency key (`matchup_key`).
    2. Map first/second Pokemon attributes from lookup.
    3. Validate join integrity.
    4. Engineer relative and interaction-based features.
    """
    base = combats_frame[["First_pokemon", "Second_pokemon", "Winner"]].copy()
    base["first_wins"] = (base["Winner"] == base["First_pokemon"]).astype(int)

    # Unordered matchup key for dependency-aware split.
    base["matchup_key"] = base.apply(
        lambda r: f"{min(r['First_pokemon'], r['Second_pokemon'])}_{max(r['First_pokemon'], r['Second_pokemon'])}",
        axis=1,
    )

    fields = STATS_COLS + ["stats_total", "Type 1", "Type 2", "Generation", "Legendary", "is_mega"]

    for prefix, id_col in [("first", "First_pokemon"), ("second", "Second_pokemon")]:
        mapped = pokedex_lookup.loc[:, fields].reindex(base[id_col]).reset_index(drop=True)
        mapped.columns = [f"{prefix}_{c}" for c in mapped.columns]
        base = pd.concat([base.reset_index(drop=True), mapped], axis=1)

    # Integrity control after join.
    join_cols = [f"first_{c}" for c in fields] + [f"second_{c}" for c in fields]
    missing_join_rows = int(base[join_cols].isna().any(axis=1).sum())
    if missing_join_rows > 0:
        raise ValueError(
            f"Detected {missing_join_rows} rows with incomplete joins. Check ID coverage in the Pokedex."
        )

    # Relative continuous features.
    for col in STATS_COLS + ["stats_total"]:
        safe_col = col.lower().replace(". ", "_").replace(" ", "_")
        base[f"diff_{safe_col}"] = base[f"first_{col}"] - base[f"second_{col}"]
        base[f"abs_diff_{safe_col}"] = base[f"diff_{safe_col}"].abs()
        base[f"first_has_adv_{safe_col}"] = (base[f"diff_{safe_col}"] > 0).astype(int)

    # Relative binary/contextual features.
    base["diff_generation"] = base["first_Generation"] - base["second_Generation"]
    base["diff_legendary"] = base["first_Legendary"] - base["second_Legendary"]
    base["diff_is_mega"] = base["first_is_mega"] - base["second_is_mega"]

    base["same_type1"] = (base["first_Type 1"] == base["second_Type 1"]).astype(int)
    base["same_type2"] = (base["first_Type 2"] == base["second_Type 2"]).astype(int)
    base["same_generation"] = (base["first_Generation"] == base["second_Generation"]).astype(int)
    base["both_legendary"] = ((base["first_Legendary"] == 1) & (base["second_Legendary"] == 1)).astype(int)

    # Relational categorical interactions.
    base["type1_pair"] = [
        _make_unordered_pair(a, b) for a, b in zip(base["first_Type 1"], base["second_Type 1"])
    ]
    base["type2_pair"] = [
        _make_unordered_pair(a, b) for a, b in zip(base["first_Type 2"], base["second_Type 2"])
    ]

    return base

### Feature engineering functions interpretation

- These functions translate EDA evidence into battle variables: differences, absolute magnitudes, and opponent relationships.
- Join integrity validation protects dataset quality before splitting and prevents silent modeling errors.

## Build final prepared dataset


In [ ]:
pokedex_lookup = build_pokedex_lookup(pokemon_df)
battle_df = build_battle_level_dataset(combats_df, pokedex_lookup)

print(f"battle_df shape: {battle_df.shape}")
display(battle_df.head(3))


### Final prepared dataset interpretation

- The final shape of `battle_df` confirms that the modeling unit is the battle, not the isolated Pokemon.
- This output is the bridge between EDA and the pipeline: variables are materialized here before typing and transformation.

## Target definition and dependency-aware split
The split is performed before any transformation fitting, avoiding leakage from repeated matchups.


In [ ]:
X = battle_df.drop(columns=[TARGET_COL])
y = battle_df[TARGET_COL].astype(int)

# Group-based split by matchup (unordered pair) to reduce train-test dependency
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=X["matchup_key"]))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Check for no group overlap between train and test
train_groups = set(X_train["matchup_key"])
test_groups = set(X_test["matchup_key"])
group_overlap = len(train_groups.intersection(test_groups))

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Target train mean: {y_train.mean():.4f}")
print(f"Target test mean: {y_test.mean():.4f}")
print(f"Group overlap (must be 0): {group_overlap}")


### Group-aware split interpretation

- `group_overlap = 0` confirms there is no leakage from repeated pairs between train and test.
- This decision operationalizes the EDA methodological warning about dependency across duplicated or related battles.

## Feature selection and typing
Identifiers and leakage-prone columns are excluded.


In [ ]:
id_like_cols = ["First_pokemon", "Second_pokemon", "Winner", "matchup_key"]
leakage_cols = sorted(set(GLOBAL_CONFIG["leakage_columns"]).intersection(X_train.columns))
drop_cols = [c for c in id_like_cols + leakage_cols if c in X_train.columns]

X_train_fe = X_train.drop(columns=drop_cols, errors="ignore").copy()
X_test_fe = X_test.drop(columns=drop_cols, errors="ignore").copy()

numeric_features = [
    c
    for c in X_train_fe.columns
    if c.startswith("diff_")
    or c.startswith("abs_diff_")
    or c.startswith("first_has_adv_")
    or c in ["first_stats_total", "second_stats_total"]
]

categorical_features = [
    c
    for c in [
        "first_Type 1",
        "second_Type 1",
        "first_Type 2",
        "second_Type 2",
        "first_Generation",
        "second_Generation",
        "type1_pair",
        "type2_pair",
    ]
    if c in X_train_fe.columns
]

binary_features = [
    c
    for c in [
        "first_Legendary",
        "second_Legendary",
        "first_is_mega",
        "second_is_mega",
        "same_type1",
        "same_type2",
        "same_generation",
        "both_legendary",
    ]
    if c in X_train_fe.columns
]

feature_decisions_df = pd.DataFrame(
    {
        "feature_group": ["dropped", "numeric", "categorical", "binary"],
        "n_features": [len(drop_cols), len(numeric_features), len(categorical_features), len(binary_features)],
        "feature_examples": [
            ", ".join(drop_cols[:8]),
            ", ".join(numeric_features[:8]),
            ", ".join(categorical_features[:8]),
            ", ".join(binary_features[:8]),
        ],
        "rationale": [
            "Identifiers/leakage-prone columns excluded before preprocessing.",
            "Relative advantage signals for battle-level predictive structure.",
            "Type/generation context and pairwise relational interactions.",
            "Low-cardinality context flags with direct tactical meaning.",
        ],
    }
)

print("Total features after cleaning:", X_train_fe.shape[1])
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Binary:", len(binary_features))
display(feature_decisions_df)

### Feature selection and typing interpretation

- The grouped summary (`numeric`, `categorical`, `binary`) explicitly justifies what is included and why.
- This traceability supports project governance: inclusion and exclusion decisions are stored and inherited by the final manifest.

## EDA questions resolved in Data Preparation
This block converts the three guiding EDA questions into explicit, testable checks for the preparation phase.

1. Structural usability for supervised learning.
2. Signal relevance for prediction (train-only evidence).
3. Descriptive-only variables excluded from the trainable feature space.

In [ ]:
from sklearn.metrics import roc_auc_score

# Q1) Structural usability for supervised learning.
q1_summary = {
    "battle_rows_after_dedup": int(len(battle_df)),
    "target_binary_train": bool(set(y_train.unique()).issubset({0, 1})),
    "target_binary_test": bool(set(y_test.unique()).issubset({0, 1})),
    "group_overlap": int(group_overlap),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
}

# Q2) Relevant predictive signals (train-only, leakage-safe).
relevance_candidates = [
    c
    for c in numeric_features
    if c.startswith("diff_") or c.startswith("abs_diff_") or c.startswith("first_has_adv_")
]

signal_rows = []
for c in relevance_candidates:
    x = X_train_fe[c]
    if x.nunique(dropna=True) <= 1:
        continue
    auc = max(roc_auc_score(y_train, x), 1 - roc_auc_score(y_train, x))
    corr = x.corr(y_train)
    signal_rows.append({"feature": c, "auc_train_oriented": auc, "corr_train": corr})

signal_ranking_df = (
    pd.DataFrame(signal_rows)
    .sort_values(["auc_train_oriented", "corr_train"], ascending=False)
    .reset_index(drop=True)
)

# Q3) Descriptive-only variables excluded from trainable space.
descriptive_only = ["Winner", "WinRate", "Wins", "n_combats", "Name", "#", "First_pokemon", "Second_pokemon"]
present_after_selection = [c for c in descriptive_only if c in X_train_fe.columns]

q3_summary = {
    "descriptive_only_detected_after_selection": present_after_selection,
    "all_descriptive_only_removed": len(present_after_selection) == 0,
}

print("Q1 - Structural usability")
print(q1_summary)
print("\nQ2 - Top train-only relational signals")
display(signal_ranking_df.head(12).round(4))
print("\nQ3 - Descriptive-only exclusion")
print(q3_summary)

assert q1_summary["group_overlap"] == 0, "Dependency leakage detected: non-zero group overlap"
assert q3_summary["all_descriptive_only_removed"], "Descriptive-only/leakage columns remain in trainable space"

### EDA question-resolution interpretation

- `Q1` validates that the problem is trainable without structural leakage.
- `Q2` confirms, in train-only evidence, which relational signals are truly strong for prediction.
- `Q3` verifies that descriptive or leakage variables do not enter the trainable space, closing methodological consistency between EDA and preparation.

## Preprocessing architecture
`ColumnTransformer` processes each feature type with the appropriate strategy.


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin", "passthrough", binary_features),
    ],
    remainder="drop",
)

preprocess_pipeline


### Preprocessing architecture interpretation

- Separation by variable type avoids incorrect transformations and improves maintainability.
- This design leaves the stage ready for reproducible training with any downstream model without rewriting cleaning logic.

## Pipeline technical validation
Fit only on train and transform train/test to verify consistency.


In [ ]:
X_train_transformed = preprocess_pipeline.fit_transform(X_train_fe)
X_test_transformed = preprocess_pipeline.transform(X_test_fe)

print("Transformed train shape:", X_train_transformed.shape)
print("Transformed test shape:", X_test_transformed.shape)
print("Nulls in y_train:", int(y_train.isna().sum()))
print("Nulls in y_test:", int(y_test.isna().sum()))


### Technical validation interpretation

- Fitting only on train and transforming test without errors confirms the pipeline is deployable and leakage-safe.
- Dimensional consistency across train/test enables direct handoff to the modeling notebook.

## Final matrices for modeling
Expose base objects for the next training phase.


In [ ]:
X_train_final = X_train_fe.copy()
X_test_final = X_test_fe.copy()

print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)


### Final matrices interpretation

- `X_train_final` and `X_test_final` are the stable tabular base for model experimentation.
- This explicit separation improves traceability: preprocessing remains decoupled from training.

## Artifact persistence
Save pipeline, split, and feature manifest for reproducibility.


## Reproducibility checklist (execution-safe)
This block validates structural consistency before handing artifacts to modeling.

In [ ]:
assert set(y_train.unique()).issubset({0, 1}), "y_train is not binary"
assert set(y_test.unique()).issubset({0, 1}), "y_test is not binary"
assert group_overlap == 0, "Group overlap detected between train and test"

assert X_train_final.shape[1] == X_test_final.shape[1], "Train/test feature count mismatch"
assert len(numeric_features) > 0, "No numeric features selected"
assert len(categorical_features) > 0, "No categorical features selected"

if hasattr(X_train_transformed, "shape") and hasattr(X_test_transformed, "shape"):
    assert X_train_transformed.shape[1] == X_test_transformed.shape[1], "Transformed train/test width mismatch"

print("Checklist OK: data preparation is reproducible and structurally consistent")

### Reproducibility checklist interpretation

- If this block passes, preparation meets minimum quality conditions for reliable training.
- It acts as a final gate: inconsistent artifacts are blocked from entering the rest of the workflow.

In [ ]:
from joblib import dump

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

dump(preprocess_pipeline, ARTIFACTS_DIR / "preprocess_pipeline_pokemon.joblib")
dump((X_train_final, X_test_final, y_train, y_test), ARTIFACTS_DIR / "split_data_pokemon.joblib")

feature_manifest = {
    "target": TARGET_COL,
    "global_config": GLOBAL_CONFIG,
    "drop_cols": drop_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "binary_features": binary_features,
    "feature_decisions": feature_decisions_df.to_dict(orient="records"),
    "notes": [
        "Winner/WinRate/Wins/n_combats are excluded as features due to leakage.",
        "Exact duplicate battles were removed before feature engineering.",
        "Type 2 is treated as structural absence with value None.",
        "Relational pair features (type1_pair/type2_pair) are included for interaction modeling.",
    ],
}

with open(ARTIFACTS_DIR / "feature_manifest_pokemon.json", "w", encoding="utf-8") as f:
    json.dump(feature_manifest, f, ensure_ascii=False, indent=2)

print("Artifacts saved in:", ARTIFACTS_DIR.resolve())

### Artifact persistence interpretation

- Files saved in `artifacts/` are technical contracts for the modeling stage (pipeline, split, and manifest).
- Unlike `reports/`, this folder stores executable operational objects; that is why this block connects directly to the training and model-selection notebook.

## Output summary
This notebook produces:
- `X_train_final`, `X_test_final`, `y_train`, `y_test`
- `preprocess_pipeline` (fitted on train only)
- `feature_decisions_df` (explicit feature-selection rationale)
- `signal_ranking_df` (train-only relevance ranking for relational features)
- `../artifacts/preprocess_pipeline_pokemon.joblib`
- `../artifacts/split_data_pokemon.joblib`
- `../artifacts/feature_manifest_pokemon.json`

How this notebook resolves the EDA guiding questions:
- Q1 (structural usability): validated through binary target checks, dependency-aware split, and zero group overlap.
- Q2 (relevant predictive signals): quantified using train-only ranking over relational features (`diff_*`, `abs_diff_*`, `first_has_adv_*`).
- Q3 (descriptive-only features): enforced by explicit exclusion of leakage/descriptive columns from trainable matrices.

Professional decisions applied in this version:
- Relational feature engineering based on interaction between contenders.
- Type interaction features (`type1_pair`, `type2_pair`) to capture matchup structure.
- Dependency-aware split by battle pair (`matchup_key`) to reduce dependency leakage.
- Explicit check for no group overlap between train and test.
- Join-integrity guard to detect rows without Pokedex mapping.
- Feature-selection rationale table persisted to artifact manifest.
- Final reproducibility checklist before handoff to modeling.